In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt
import seaborn as sns

movies = pd.read_csv("../data/movies.csv")
rating = pd.read_csv("../data/ratings.csv")
#print(f"movies : \n {movies.head(1)}")
#print(f"rating : \n {rating.head(1)}")

final_dataset = rating.pivot(index="movieId", columns="userId", values="rating")
final_dataset.fillna(0, inplace=True)
no_user_voted = rating.groupby("movieId")["rating"].agg("count")
#print(no_user_voted)
no_movies_voted = rating.groupby("userId")["rating"].agg("count")
#print(no_movies_voted)


#now keep the movies that where voted by more than 10 users.
final_dataset=final_dataset.loc[no_user_voted[no_user_voted > 10].index, no_movies_voted[no_movies_voted > 50].index]




csr_data = csr_matrix(final_dataset.values)
final_dataset.reset_index(inplace=True)

knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=20, n_jobs=-1)
knn.fit(csr_data) #fit the sparse data


def get_movie_recommendation(movie_name):
    n_movies_to_reccomend = 10
    movie_list = movies[movies['title'].str.contains(movie_name)]  
    if len(movie_list):        
        movie_idx= movie_list.iloc[0]['movieId']
        movie_idx = final_dataset[final_dataset['movieId'] == movie_idx].index[0]
        
        distances , indices = knn.kneighbors(csr_data[movie_idx],n_neighbors=n_movies_to_reccomend+1)    
        rec_movie_indices = sorted(list(zip(indices.squeeze().tolist(),distances.squeeze().tolist())),\
                               key=lambda x: x[1])[:0:-1]
        
        recommend_frame = []
        
        for val in rec_movie_indices:
            movie_idx = final_dataset.iloc[val[0]]['movieId']
            idx = movies[movies['movieId'] == movie_idx].index
            recommend_frame.append({'Title':movies.iloc[idx]['title'].values[0],'Distance':val[1]})
        df = pd.DataFrame(recommend_frame,index=range(1,n_movies_to_reccomend+1))
        return df
    
    else:
        
        return "No movies found. Please check your input"
    


In [5]:
get_movie_recommendation('Finding Nemo')

,Title,Distance
1,Ocean's Eleven (2001),0.381464
2,Kill Bill: Vol. 1 (2003),0.375562
3,"Lord of the Rings: The Two Towers, The (2002)",0.364964
4,Catch Me If You Can (2002),0.352445
5,Shrek 2 (2004),0.346279
6,"Lord of the Rings: The Return of the King, The...",0.335545
7,Pirates of the Caribbean: The Curse of the Bla...,0.295604
8,"Monsters, Inc. (2001)",0.278293
9,Shrek (2001),0.256021
10,"Incredibles, The (2004)",0.241599
